# Time series generation ONLINE


In [ ]:
# BPIC2012 CUTOFF TEST
# Process inputs: UTC dates, compared with the saved daily CSV.
# Actor labels/features: Amsterdam local dates, cutoff run compared with a
# full-log reference built from the same XES events.

import sys, subprocess, importlib.util, zipfile, shutil, re
from pathlib import Path
from bisect import bisect_left, bisect_right, insort

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from google.colab import files

if importlib.util.find_spec("pm4py") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pm4py"])
import pm4py

def is_log_file(name):
    name = name.lower()
    return (
        name.endswith((".xes", ".zip"))
        or re.search(r"\.xes(?: \(\d+\))?\.gz$", name) is not None
    )

print("Upload the saved BPIC2012 daily time-series CSV.")
daily_upload = files.upload()
daily_files = [n for n in daily_upload if n.lower().endswith(".csv")]
if len(daily_files) != 1:
    raise ValueError("Upload exactly one daily time-series CSV.")

daily = pd.read_csv(daily_files[0], index_col=0, parse_dates=True)
daily.index = pd.to_datetime(daily.index).tz_localize(None).normalize()

print("Upload the original BPIC2012 XES, XES.GZ, or ZIP event log.")
log_upload = files.upload()
log_files = [n for n in log_upload if is_log_file(n)]
if len(log_files) != 1:
    raise ValueError(f"Upload exactly one XES, XES.GZ, or ZIP file. Found: {log_files}")

log_path = log_files[0]
if log_path.lower().endswith(".zip"):
    with zipfile.ZipFile(log_path) as zf:
        members = [
            n for n in zf.namelist()
            if is_log_file(n) and not n.lower().endswith(".zip")
        ]
        if len(members) != 1:
            raise ValueError(f"Expected exactly one XES/XES.GZ in ZIP; found {members[:10]}")
        log_path = str(Path("/content/bpic2012_cutoff_test") / Path(members[0]).name)
        Path(log_path).parent.mkdir(parents=True, exist_ok=True)
        with zf.open(members[0]) as src, open(log_path, "wb") as dst:
            shutil.copyfileobj(src, dst)

print("Reading XES events...")
xes = pm4py.convert_to_dataframe(pm4py.read_xes(log_path))

CASE, EVENT, TIME = "case:concept:name", "concept:name", "time:timestamp"
ACTOR = next((c for c in ["org:resource", "org:group"] if c in xes.columns), None)
if ACTOR is None:
    raise KeyError("XES needs an org:resource or org:group column.")

# Process-input timeline uses UTC, matching the saved process columns.
process = xes[[CASE, EVENT, TIME, ACTOR]].copy()
process[TIME] = pd.to_datetime(
    process[TIME], errors="coerce", utc=True
).dt.tz_localize(None)
process = process.dropna(subset=[CASE, TIME]).copy()
process[CASE] = process[CASE].astype(str)
process["event_day"] = process[TIME].dt.normalize()

# Actor classification and aggregation use Amsterdam local clock dates,
# matching the date convention in the enriched behavior CSV.
actor_events = xes[[CASE, EVENT, TIME, ACTOR]].copy()
actor_events[TIME] = (
    pd.to_datetime(actor_events[TIME], errors="coerce", utc=True)
    .dt.tz_convert("Europe/Amsterdam")
    .dt.tz_localize(None)
)
actor_events = actor_events.dropna(subset=[CASE, EVENT, TIME, ACTOR]).copy()
actor_events[CASE] = actor_events[CASE].astype(str)
actor_events = actor_events.rename(columns={
    CASE: "case_id", EVENT: "event", TIME: "timestamp", ACTOR: "actor"
})
actor_events["event"] = actor_events["event"].astype(str)
actor_events["actor"] = actor_events["actor"].astype(str)

def classify_at_cutoff(events):
    """Apply the actor-behavior classification logic to the available events."""
    events = events.sort_values(
        ["case_id", "timestamp"], kind="mergesort"
    ).reset_index(drop=True)

    case_ids = events["case_id"].to_numpy()
    event_names = events["event"].to_numpy()
    actors = events["actor"].to_numpy()
    times = events["timestamp"].to_numpy()

    actor_history = {}
    result = []

    for i in range(len(events) - 1):
        if case_ids[i] != case_ids[i + 1]:
            continue

        r_i, r_j = actors[i], actors[i + 1]
        t_i, t_j = times[i], times[i + 1]

        check_actor = r_i if r_i == r_j else r_j
        history = actor_history.get(check_actor, [])
        intervening = bisect_left(history, t_j) > bisect_right(history, t_i)

        if r_i == r_j:
            behavior = "I" if intervening else "C"
        else:
            behavior = "HB" if intervening else "HI"

        result.append({
            "case_id": case_ids[i],
            "event_i": event_names[i],
            "timestamp_i": t_i,
            "event_j": event_names[i + 1],
            "timestamp_j": t_j,
            "actor_i": r_i,
            "actor_j": r_j,
            "behavior": behavior,
            "delta_t": (t_j - t_i) / np.timedelta64(1, "s"),
        })

        insort(actor_history.setdefault(r_i, []), t_i)
        insort(actor_history.setdefault(r_j, []), t_j)

    return pd.DataFrame(result)

print(f"Process events: {len(process):,}")
print(f"Actor-complete events: {len(actor_events):,}")
print("Classifying the full XES log for the reference...")
full_transitions = classify_at_cutoff(actor_events)
print(f"Full-log transitions: {len(full_transitions):,}")

label_key = [
    "case_id", "event_i", "timestamp_i", "event_j",
    "timestamp_j", "actor_i", "actor_j"
]
behaviors = ["C", "I", "HI", "HB"]

label_mismatches_total = 0
actor_mismatches_vs_full = {
    col: 0
    for b in behaviors
    for col in [f"Cum_Count_{b}", f"Cum_Time_{b}_seconds"]
}
cutoff_rows = []

for day in tqdm(daily.index, desc="Rebuilding daily cutoffs"):
    cutoff = day + pd.Timedelta(days=1)
    prefix = actor_events.loc[actor_events["timestamp"] < cutoff]
    cutoff_transitions = classify_at_cutoff(prefix)

    # Check whether labels for available transitions change when future events
    # are hidden.
    if len(cutoff_transitions):
        label_check = cutoff_transitions[label_key + ["behavior"]].merge(
            full_transitions[label_key + ["behavior"]],
            on=label_key,
            how="left",
            suffixes=("_cutoff", "_full"),
        )
        label_mismatch = int(
            (
                label_check["behavior_full"].notna()
                & (label_check["behavior_cutoff"] != label_check["behavior_full"])
            ).sum()
        )
    else:
        label_mismatch = 0

    label_mismatches_total += label_mismatch
    row = {"cutoff_date": day, "label_mismatches": label_mismatch}

    # Compare cutoff features with full-XES features through the same local date.
    full_available = full_transitions[
        pd.to_datetime(full_transitions["timestamp_j"]).dt.normalize() <= day
    ] if len(full_transitions) else full_transitions

    for behavior in behaviors:
        if len(cutoff_transitions):
            cutoff_days = pd.to_datetime(cutoff_transitions["timestamp_j"]).dt.normalize()
            cutoff_selected = cutoff_transitions[
                (cutoff_transitions["behavior"] == behavior)
                & (cutoff_days <= day)
            ]
        else:
            cutoff_selected = cutoff_transitions

        full_selected = full_available[
            full_available["behavior"] == behavior
        ]

        count_col = f"Cum_Count_{behavior}"
        time_col = f"Cum_Time_{behavior}_seconds"

        cutoff_count = len(cutoff_selected)
        full_count = len(full_selected)
        cutoff_seconds = (
            float(cutoff_selected["delta_t"].sum()) if len(cutoff_selected) else 0.0
        )
        full_seconds = (
            float(full_selected["delta_t"].sum()) if len(full_selected) else 0.0
        )

        row[count_col] = cutoff_count
        row[f"full_{count_col}"] = full_count
        row[time_col] = cutoff_seconds
        row[f"full_{time_col}"] = full_seconds

        if cutoff_count != full_count:
            actor_mismatches_vs_full[count_col] += 1
        if not np.isclose(cutoff_seconds, full_seconds, atol=1e-3):
            actor_mismatches_vs_full[time_col] += 1

    cutoff_rows.append(row)

cutoff_results = pd.DataFrame(cutoff_rows)

# Rebuild process inputs on UTC dates and compare them with the saved CSV.
process["case_start"] = process.groupby(CASE)[TIME].transform("min")
process["elapsed_hours_at_eod"] = (
    (process["event_day"] + pd.Timedelta(days=1) - process["case_start"])
    .dt.total_seconds() / 3600.0
)
case_day = process[
    [CASE, "event_day", "case_start", "elapsed_hours_at_eod"]
].drop_duplicates([CASE, "event_day"])

process_daily = process.groupby("event_day").agg(
    Events_During_Day=(EVENT, "size"),
    Active_Resources_During_Day=(ACTOR, "nunique"),
)
process_daily["Cases_With_Activity"] = process.groupby("event_day")[CASE].nunique()
process_daily["Avg_Elapsed_Time"] = (
    case_day.groupby("event_day")["elapsed_hours_at_eod"].mean()
)

process_mismatches = {}
for col in [
    "Events_During_Day",
    "Active_Resources_During_Day",
    "Cases_With_Activity",
    "Avg_Elapsed_Time",
]:
    if col in daily.columns:
        expected = process_daily[col].reindex(daily.index)
        actual = pd.to_numeric(daily[col], errors="coerce")
        process_mismatches[col] = int(
            (~np.isclose(actual.fillna(0), expected.fillna(0), atol=1e-6)).sum()
        )

print("\nCUTOFF TEST RESULTS")
print(f"Label mismatches against full-XES labels: {label_mismatches_total:,}")
print(f"Daily dates tested: {len(daily):,}")

print("\nProcess-input mismatches against saved CSV:")
display(pd.DataFrame([
    {"column": col, "mismatching_dates": count}
    for col, count in process_mismatches.items()
]).sort_values("mismatching_dates", ascending=False))

print("\nActor-feature mismatches: cutoff run vs full-XES reference")
display(pd.DataFrame([
    {"column": col, "mismatching_dates": count}
    for col, count in actor_mismatches_vs_full.items()
]).sort_values("mismatching_dates", ascending=False))

print("\nFirst cutoff results:")
display(cutoff_results.head(10))

all_checks_pass = (
    label_mismatches_total == 0
    and sum(process_mismatches.values()) == 0
    and sum(actor_mismatches_vs_full.values()) == 0
)

print("\nPASS" if all_checks_pass else "\nFAIL")



  Welcome to PM4Py — Community Version
  Open-Source License (AGPL v3)

  📚 Docs & Examples:
     https://processintelligence.solutions/pm4py

  ⚖️  License: AGPL v3 — Commercial use requires open-sourcing your application.
     Business use without open-sourcing? A commercial license is available:
     https://processintelligence.solutions/pm4py#licensing




Upload the saved BPIC2012 daily time-series CSV.


Saving BPIC2012_final_online_daily_time_series_cutoff_safe.csv to BPIC2012_final_online_daily_time_series_cutoff_safe.csv
Upload the original BPIC2012 XES, XES.GZ, or ZIP event log.


Saving BPI_Challenge_2012.xes.gz to BPI_Challenge_2012.xes.gz
Reading XES events...


/usr/local/lib/python3.13/dist-packages/pm4py/utils.py:1027: UserWarning: Install the optional requirement `r4pm` to import/export files faster. `rustxes` remains supported as a fallback.
  warnings.warn(


parsing log, completed traces ::   0%|          | 0/13087 [00:00<?, ?it/s]

... (5 outputs left out)

,column,mismatching_dates
0,Cum_Count_C,0
1,Cum_Time_C_seconds,0
2,Cum_Count_I,0
3,Cum_Time_I_seconds,0
4,Cum_Count_HI,0
5,Cum_Time_HI_seconds,0
6,Cum_Count_HB,0
7,Cum_Time_HB_seconds,0



First cutoff results:


,cutoff_date,label_mismatches,Cum_Count_C,full_Cum_Count_C,Cum_Time_C_seconds,full_Cum_Time_C_seconds,Cum_Count_I,full_Cum_Count_I,Cum_Time_I_seconds,full_Cum_Time_I_seconds,Cum_Count_HI,full_Cum_Count_HI,Cum_Time_HI_seconds,full_Cum_Time_HI_seconds,Cum_Count_HB,full_Cum_Count_HB,Cum_Time_HB_seconds,full_Cum_Time_HB_seconds
0,2011-09-30,0,0,0,0.000,0.000,0,0,0.000,0.000,0,0,0.000000e+00,0.000000e+00,0,0,0.000000e+00,0.000000e+00
1,2011-10-01,0,255,255,25669.799,25669.799,5,5,10093.963,10093.963,18,18,1.009155e+05,1.009155e+05,18,18,9.255140e+04,9.255140e+04
2,2011-10-02,0,368,368,27286.354,27286.354,7,7,10170.240,10170.240,18,18,1.009155e+05,1.009155e+05,18,18,9.255140e+04,9.255140e+04
3,2011-10-03,0,1242,1242,109022.232,109022.232,26,26,49907.369,49907.369,92,92,2.491017e+06,2.491017e+06,112,112,3.355609e+06,3.355609e+06
4,2011-10-04,0,1987,1987,180984.494,180984.494,44,44,153262.395,153262.395,171,171,3.773225e+06,3.773225e+06,209,209,6.277360e+06,6.277360e+06
5,2011-10-05,0,2958,2958,275890.638,275890.638,76,76,254327.029,254327.029,247,247,5.247898e+06,5.247898e+06,309,309,7.959022e+06,7.959022e+06
6,2011-10-06,0,3617,3617,357067.015,357067.015,87,87,424004.919,424004.919,301,301,7.704139e+06,7.704139e+06,391,391,1.069459e+07,1.069459e+07
7,2011-10-07,0,4387,4387,569484.621,569484.621,107,107,502132.339,502132.339,361,361,9.379056e+06,9.379056e+06,494,494,1.496213e+07,1.496213e+07
8,2011-10-08,0,4764,4764,1261908.899,1261908.899,127,127,1284307.293,1284307.293,396,396,1.187747e+07,1.187747e+07,529,529,1.719175e+07,1.719175e+07
9,2011-10-09,0,4922,4922,1264023.989,1264023.989,128,128,1284338.197,1284338.197,396,396,1.187747e+07,1.187747e+07,529,529,1.719175e+07,1.719175e+07



PASS


In [ ]:
# BPIC2017 CUTOFF TEST — compiled classification
# Process inputs: UTC dates, compared with the saved daily CSV.
# Actor labels/features: Amsterdam local dates.
# Each cutoff is classified independently using only available events.

import sys
import subprocess
import importlib.util
import zipfile
import shutil
import re

from pathlib import Path

import numpy as np
import pandas as pd

from tqdm.auto import tqdm
from google.colab import files
from IPython.display import display

for package in ["pm4py", "numba"]:
    if importlib.util.find_spec(package) is None:
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", package
        ])

import pm4py
from numba import njit


def is_log_file(name):
    name = name.lower()
    return (
        name.endswith((".xes", ".zip"))
        or re.search(r"\.xes(?: \(\d+\))?\.gz$", name) is not None
    )


TRANSITION_COLUMNS = [
    "case_id",
    "event_i",
    "timestamp_i",
    "event_j",
    "timestamp_j",
    "actor_i",
    "actor_j",
    "behavior",
    "delta_t",
]


@njit
def _classify_history(
    case_codes,
    actor_codes,
    times,
    sorted_times,
    positions,
    offsets,
):
    n = len(times)

    # Fenwick tree: counts of history entries inserted so far.
    tree = np.zeros(n + 1, dtype=np.int64)
    labels = np.full(max(0, n - 1), -1, dtype=np.int8)

    for i in range(n - 1):
        if case_codes[i] != case_codes[i + 1]:
            continue

        # In the original logic, check_actor is always actor_j.
        actor_j = actor_codes[i + 1]
        start = offsets[actor_j]
        end = offsets[actor_j + 1]
        actor_times = sorted_times[start:end]

        # Strict interval: time_i < history_timestamp < time_j.
        low = start + np.searchsorted(
            actor_times, times[i], side="right"
        )
        high = start + np.searchsorted(
            actor_times, times[i + 1], side="left"
        )

        count = 0

        k = high
        while k > 0:
            count += tree[k]
            k -= k & -k

        k = low
        while k > 0:
            count -= tree[k]
            k -= k & -k

        same_actor = actor_codes[i] == actor_j

        # C=0, I=1, HI=2, HD=3.
        labels[i] = (
            (0 if same_actor else 2)
            + (1 if count > 0 else 0)
        )

        # Preserve both original endpoint insertions, including duplicates.
        k = positions[i] + 1
        while k <= n:
            tree[k] += 1
            k += k & -k

        k = positions[i + 1] + 1
        while k <= n:
            tree[k] += 1
            k += k & -k

    return labels


def classify_at_cutoff(events):
    """Classify events already sorted by case_id and timestamp."""
    n = len(events)

    if n < 2:
        return pd.DataFrame(columns=TRANSITION_COLUMNS)

    case_ids = events["case_id"].to_numpy()
    event_names = events["event"].to_numpy()
    actors = events["actor"].to_numpy()
    timestamps = events["timestamp"].to_numpy(dtype="datetime64[ns]")
    times = timestamps.astype(np.int64)

    case_codes = pd.factorize(
        case_ids, sort=False
    )[0].astype(np.int64)

    actor_codes = pd.factorize(
        actors, sort=False
    )[0].astype(np.int64)

    # Coordinate positions are ordered by actor, then timestamp.
    order = np.lexsort((times, actor_codes))
    sorted_times = times[order]

    positions = np.empty(n, dtype=np.int64)
    positions[order] = np.arange(n, dtype=np.int64)

    sizes = np.bincount(actor_codes)
    offsets = np.empty(len(sizes) + 1, dtype=np.int64)
    offsets[0] = 0
    offsets[1:] = np.cumsum(sizes)

    # History counts start empty for every independent classification.
    labels = _classify_history(
        case_codes,
        actor_codes,
        times,
        sorted_times,
        positions,
        offsets,
    )

    i = np.flatnonzero(labels >= 0)
    j = i + 1
    behavior_names = np.array(["C", "I", "HI", "HD"])

    return pd.DataFrame({
        "case_id": case_ids[i],
        "event_i": event_names[i],
        "timestamp_i": timestamps[i],
        "event_j": event_names[j],
        "timestamp_j": timestamps[j],
        "actor_i": actors[i],
        "actor_j": actors[j],
        "behavior": behavior_names[labels[i]],
        "delta_t": (
            (timestamps[j] - timestamps[i])
            / np.timedelta64(1, "s")
        ),
    })


print("Upload the saved BPIC2017 daily time-series CSV.")
daily_upload = files.upload()
daily_files = [
    name for name in daily_upload
    if name.lower().endswith(".csv")
]

if len(daily_files) != 1:
    raise ValueError("Upload exactly one daily time-series CSV.")

daily = pd.read_csv(
    daily_files[0], index_col=0, parse_dates=True
)
daily.index = (
    pd.to_datetime(daily.index)
    .tz_localize(None)
    .normalize()
)

print("Upload the original BPIC2017 XES, XES.GZ, or ZIP event log.")
log_upload = files.upload()
log_files = [
    name for name in log_upload
    if is_log_file(name)
]

if len(log_files) != 1:
    raise ValueError(
        "Upload exactly one XES, XES.GZ, or ZIP file. "
        f"Found: {log_files}"
    )

log_path = log_files[0]

if log_path.lower().endswith(".zip"):
    with zipfile.ZipFile(log_path) as zf:
        members = [
            name for name in zf.namelist()
            if is_log_file(name)
            and not name.lower().endswith(".zip")
        ]

        if len(members) != 1:
            raise ValueError(
                "Expected exactly one XES/XES.GZ in ZIP; "
                f"found {members[:10]}"
            )

        log_path = str(
            Path("/content/bpic2017_cutoff_test")
            / Path(members[0]).name
        )
        Path(log_path).parent.mkdir(
            parents=True, exist_ok=True
        )

        with zf.open(members[0]) as src:
            with open(log_path, "wb") as dst:
                shutil.copyfileobj(src, dst)

print("Reading XES events...")
xes = pm4py.convert_to_dataframe(
    pm4py.read_xes(log_path)
)

CASE = "case:concept:name"
EVENT = "concept:name"
TIME = "time:timestamp"

ACTOR = next(
    (
        column
        for column in ["org:resource", "org:group"]
        if column in xes.columns
    ),
    None,
)

if ACTOR is None:
    raise KeyError(
        "XES needs an org:resource or org:group column."
    )

# Process-input timeline uses UTC.
process = xes[[CASE, EVENT, TIME, ACTOR]].copy()
process[TIME] = pd.to_datetime(
    process[TIME], errors="coerce", utc=True
).dt.tz_localize(None)

process = process.dropna(
    subset=[CASE, TIME]
).copy()
process[CASE] = process[CASE].astype(str)
process["event_day"] = process[TIME].dt.normalize()

# Actor classification uses Amsterdam local clock dates.
actor_events = xes[[CASE, EVENT, TIME, ACTOR]].copy()
actor_events[TIME] = (
    pd.to_datetime(
        actor_events[TIME], errors="coerce", utc=True
    )
    .dt.tz_convert("Europe/Amsterdam")
    .dt.tz_localize(None)
)

actor_events = actor_events.dropna(
    subset=[CASE, EVENT, TIME, ACTOR]
).copy()
actor_events[CASE] = actor_events[CASE].astype(str)

actor_events = actor_events.rename(columns={
    CASE: "case_id",
    EVENT: "event",
    TIME: "timestamp",
    ACTOR: "actor",
})
actor_events["event"] = actor_events["event"].astype(str)
actor_events["actor"] = actor_events["actor"].astype(str)

# Sort once. Every cutoff filter preserves this order.
actor_events = actor_events.sort_values(
    ["case_id", "timestamp"], kind="mergesort"
).reset_index(drop=True)

print(f"Process events: {len(process):,}")
print(f"Actor-complete events: {len(actor_events):,}")

# Compile the numeric classifier before processing the full log.
print("Compiling the classifier (first run only)...")
_classify_history(
    np.array([0, 0], dtype=np.int64),
    np.array([0, 0], dtype=np.int64),
    np.array([0, 1], dtype=np.int64),
    np.array([0, 1], dtype=np.int64),
    np.array([0, 1], dtype=np.int64),
    np.array([0, 2], dtype=np.int64),
)

print("Classifying the full XES log for the reference...")
full_transitions = classify_at_cutoff(actor_events)
print(f"Full-log transitions: {len(full_transitions):,}")

label_key = [
    "case_id",
    "event_i",
    "timestamp_i",
    "event_j",
    "timestamp_j",
    "actor_i",
    "actor_j",
]
behaviors = ["C", "I", "HI", "HD"]

full_labels = full_transitions[
    label_key + ["behavior"]
]
full_transition_days = pd.to_datetime(
    full_transitions["timestamp_j"]
).dt.normalize()

label_mismatches_total = 0
actor_mismatches_vs_full = {
    column: 0
    for behavior in behaviors
    for column in [
        f"Cum_Count_{behavior}",
        f"Cum_Time_{behavior}_seconds",
    ]
}
cutoff_rows = []

for day in tqdm(
    daily.index, desc="Rebuilding daily cutoffs"
):
    cutoff = day + pd.Timedelta(days=1)

    # Classify this cutoff independently using only available events.
    prefix = actor_events.loc[
        actor_events["timestamp"] < cutoff
    ]
    cutoff_transitions = classify_at_cutoff(prefix)

    # Compare labels against the full-log reference.
    if len(cutoff_transitions):
        label_check = cutoff_transitions[
            label_key + ["behavior"]
        ].merge(
            full_labels,
            on=label_key,
            how="left",
            suffixes=("_cutoff", "_full"),
        )

        label_mismatch = int(
            (
                label_check["behavior_full"].notna()
                & (
                    label_check["behavior_cutoff"]
                    != label_check["behavior_full"]
                )
            ).sum()
        )
    else:
        label_mismatch = 0

    label_mismatches_total += label_mismatch

    row = {
        "cutoff_date": day,
        "label_mismatches": label_mismatch,
    }

    cutoff_days = pd.to_datetime(
        cutoff_transitions["timestamp_j"]
    ).dt.normalize()

    full_available = full_transitions.loc[
        full_transition_days <= day
    ]

    for behavior in behaviors:
        cutoff_selected = cutoff_transitions.loc[
            (cutoff_transitions["behavior"] == behavior)
            & (cutoff_days <= day)
        ]

        full_selected = full_available.loc[
            full_available["behavior"] == behavior
        ]

        count_col = f"Cum_Count_{behavior}"
        time_col = f"Cum_Time_{behavior}_seconds"

        cutoff_count = len(cutoff_selected)
        full_count = len(full_selected)

        cutoff_seconds = (
            float(cutoff_selected["delta_t"].sum())
            if len(cutoff_selected)
            else 0.0
        )
        full_seconds = (
            float(full_selected["delta_t"].sum())
            if len(full_selected)
            else 0.0
        )

        row[count_col] = cutoff_count
        row[f"full_{count_col}"] = full_count
        row[time_col] = cutoff_seconds
        row[f"full_{time_col}"] = full_seconds

        if cutoff_count != full_count:
            actor_mismatches_vs_full[count_col] += 1

        if not np.isclose(
            cutoff_seconds, full_seconds, atol=1e-3
        ):
            actor_mismatches_vs_full[time_col] += 1

    cutoff_rows.append(row)

cutoff_results = pd.DataFrame(cutoff_rows)

# Rebuild process inputs on UTC dates.
process["case_start"] = (
    process.groupby(CASE)[TIME].transform("min")
)
process["elapsed_hours_at_eod"] = (
    (
        process["event_day"]
        + pd.Timedelta(days=1)
        - process["case_start"]
    ).dt.total_seconds()
    / 3600.0
)

case_day = process[
    [
        CASE,
        "event_day",
        "case_start",
        "elapsed_hours_at_eod",
    ]
].drop_duplicates([CASE, "event_day"])

process_daily = process.groupby("event_day").agg(
    Events_During_Day=(EVENT, "size"),
    Active_Resources_During_Day=(ACTOR, "nunique"),
)
process_daily["Cases_With_Activity"] = (
    process.groupby("event_day")[CASE].nunique()
)
process_daily["Avg_Elapsed_Time"] = (
    case_day.groupby("event_day")[
        "elapsed_hours_at_eod"
    ].mean()
)

process_mismatches = {}

for column in [
    "Events_During_Day",
    "Active_Resources_During_Day",
    "Cases_With_Activity",
    "Avg_Elapsed_Time",
]:
    if column in daily.columns:
        expected = process_daily[column].reindex(
            daily.index
        )
        actual = pd.to_numeric(
            daily[column], errors="coerce"
        )

        process_mismatches[column] = int(
            (
                ~np.isclose(
                    actual.fillna(0),
                    expected.fillna(0),
                    atol=1e-6,
                )
            ).sum()
        )

print("\nCUTOFF TEST RESULTS")
print(
    "Label mismatches against full-XES labels: "
    f"{label_mismatches_total:,}"
)
print(f"Daily dates tested: {len(daily):,}")

print("\nProcess-input mismatches against saved CSV:")
display(
    pd.DataFrame(
        [
            {
                "column": column,
                "mismatching_dates": count,
            }
            for column, count in process_mismatches.items()
        ],
        columns=["column", "mismatching_dates"],
    ).sort_values(
        "mismatching_dates", ascending=False
    )
)

print(
    "\nActor-feature mismatches: "
    "cutoff run vs full-XES reference"
)
display(
    pd.DataFrame([
        {
            "column": column,
            "mismatching_dates": count,
        }
        for column, count in actor_mismatches_vs_full.items()
    ]).sort_values(
        "mismatching_dates", ascending=False
    )
)

print("\nFirst cutoff results:")
display(cutoff_results.head(10))

all_checks_pass = (
    label_mismatches_total == 0
    and sum(process_mismatches.values()) == 0
    and sum(actor_mismatches_vs_full.values()) == 0
)

print("\nPASS" if all_checks_pass else "\nFAIL")

Upload the saved BPIC2017 daily time-series CSV.


Saving BPIC2017_final_online_daily_time_series_cutoff_safe.csv to BPIC2017_final_online_daily_time_series_cutoff_safe (3).csv
Upload the original BPIC2017 XES, XES.GZ, or ZIP event log.


Saving BPI Challenge 2017.xes.zip to BPI Challenge 2017.xes (3).zip
Reading XES events...


parsing log, completed traces ::   0%|          | 0/31509 [00:00<?, ?it/s]

Process events: 1,202,267
Actor-complete events: 1,202,267
Compiling the classifier (first run only)...
Classifying the full XES log for the reference...
Full-log transitions: 1,170,758


Rebuilding daily cutoffs:   0%|          | 0/398 [00:00<?, ?it/s]

... (3 outputs left out)

,column,mismatching_dates
0,Cum_Count_C,0
1,Cum_Time_C_seconds,0
2,Cum_Count_I,0
3,Cum_Time_I_seconds,0
4,Cum_Count_HI,0
5,Cum_Time_HI_seconds,0
6,Cum_Count_HD,0
7,Cum_Time_HD_seconds,0



First cutoff results:


,cutoff_date,label_mismatches,Cum_Count_C,full_Cum_Count_C,Cum_Time_C_seconds,full_Cum_Time_C_seconds,Cum_Count_I,full_Cum_Count_I,Cum_Time_I_seconds,full_Cum_Time_I_seconds,Cum_Count_HI,full_Cum_Count_HI,Cum_Time_HI_seconds,full_Cum_Time_HI_seconds,Cum_Count_HD,full_Cum_Count_HD,Cum_Time_HD_seconds,full_Cum_Time_HD_seconds
0,2016-01-01,0,104,104,1429.517,1429.517,0,0,0.000,0.000,0,0,0.000000e+00,0.000000e+00,0,0,0.000000e+00,0.000000e+00
1,2016-01-02,0,573,573,40411.863,40411.863,3,3,24493.147,24493.147,17,17,5.328133e+05,5.328133e+05,19,19,1.046801e+06,1.046801e+06
2,2016-01-03,0,741,741,42598.611,42598.611,5,5,24660.851,24660.851,17,17,5.328133e+05,5.328133e+05,19,19,1.046801e+06,1.046801e+06
3,2016-01-04,0,2453,2453,191872.779,191872.779,23,23,98324.968,98324.968,91,91,3.136741e+06,3.136741e+06,154,154,9.477910e+06,9.477910e+06
4,2016-01-05,0,3898,3898,388413.367,388413.367,37,37,316521.957,316521.957,141,141,4.389474e+06,4.389474e+06,296,296,1.681678e+07,1.681678e+07
5,2016-01-06,0,5254,5254,495534.636,495534.636,52,52,937820.849,937820.849,191,191,7.396594e+06,7.396594e+06,411,411,2.977391e+07,2.977391e+07
6,2016-01-07,0,6511,6511,600995.529,600995.529,63,63,975208.293,975208.293,232,232,8.810695e+06,8.810695e+06,502,502,3.616501e+07,3.616501e+07
7,2016-01-08,0,7653,7653,716338.733,716338.733,80,80,3017600.485,3017600.485,299,299,1.219099e+07,1.219099e+07,703,703,7.008708e+07,7.008708e+07
8,2016-01-09,0,8675,8675,803207.892,803207.892,85,85,3775928.343,3775928.343,345,345,1.410547e+07,1.410547e+07,851,851,8.688312e+07,8.688312e+07
9,2016-01-10,0,8861,8861,805607.858,805607.858,86,86,3776003.377,3776003.377,345,345,1.410547e+07,1.410547e+07,852,852,8.690838e+07,8.690838e+07



PASS


In [ ]:
# BPIC2011 CUTOFF TEST — compiled classification
# Process inputs: Amsterdam dates, elapsed time in days.
# Elapsed-time arithmetic retains timezone information.
# Actor labels/features: Amsterdam local dates.
# Each cutoff is classified independently using only available events.

import sys
import subprocess
import importlib.util
import zipfile
import shutil
import re

from pathlib import Path

import numpy as np
import pandas as pd

from tqdm.auto import tqdm
from google.colab import files
from IPython.display import display

for package in ["pm4py", "numba"]:
    if importlib.util.find_spec(package) is None:
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", package
        ])

import pm4py
from numba import njit


def is_log_file(name):
    name = name.lower()
    return (
        name.endswith((".xes", ".zip"))
        or re.search(r"\.xes(?: ?\(\d+\))?\.gz$", name) is not None
    )


TRANSITION_COLUMNS = [
    "case_id",
    "event_i",
    "timestamp_i",
    "event_j",
    "timestamp_j",
    "actor_i",
    "actor_j",
    "behavior",
    "delta_t",
]


@njit
def _classify_history(
    case_codes,
    actor_codes,
    times,
    sorted_times,
    positions,
    offsets,
):
    n = len(times)

    # Counts of history entries inserted so far.
    tree = np.zeros(n + 1, dtype=np.int64)
    labels = np.full(max(0, n - 1), -1, dtype=np.int8)

    for i in range(n - 1):
        if case_codes[i] != case_codes[i + 1]:
            continue

        # In the original logic, check_actor is always actor_j.
        actor_j = actor_codes[i + 1]
        start = offsets[actor_j]
        end = offsets[actor_j + 1]
        actor_times = sorted_times[start:end]

        # Strict interval: time_i < history_timestamp < time_j.
        low = start + np.searchsorted(
            actor_times, times[i], side="right"
        )
        high = start + np.searchsorted(
            actor_times, times[i + 1], side="left"
        )

        count = 0

        k = high
        while k > 0:
            count += tree[k]
            k -= k & -k

        k = low
        while k > 0:
            count -= tree[k]
            k -= k & -k

        same_actor = actor_codes[i] == actor_j

        # C=0, I=1, HI=2, HB=3.
        labels[i] = (
            (0 if same_actor else 2)
            + (1 if count > 0 else 0)
        )

        # Preserve both original endpoint insertions, including duplicates.
        k = positions[i] + 1
        while k <= n:
            tree[k] += 1
            k += k & -k

        k = positions[i + 1] + 1
        while k <= n:
            tree[k] += 1
            k += k & -k

    return labels


def classify_at_cutoff(events):
    """Classify events already sorted by case_id and timestamp."""
    n = len(events)

    if n < 2:
        return pd.DataFrame(columns=TRANSITION_COLUMNS)

    case_ids = events["case_id"].to_numpy()
    event_names = events["event"].to_numpy()
    actors = events["actor"].to_numpy()
    timestamps = events["timestamp"].to_numpy(dtype="datetime64[ns]")
    times = timestamps.astype(np.int64)

    case_codes = pd.factorize(
        case_ids, sort=False
    )[0].astype(np.int64)

    actor_codes = pd.factorize(
        actors, sort=False
    )[0].astype(np.int64)

    # Coordinate positions are ordered by actor, then timestamp.
    order = np.lexsort((times, actor_codes))
    sorted_times = times[order]

    positions = np.empty(n, dtype=np.int64)
    positions[order] = np.arange(n, dtype=np.int64)

    sizes = np.bincount(actor_codes)
    offsets = np.empty(len(sizes) + 1, dtype=np.int64)
    offsets[0] = 0
    offsets[1:] = np.cumsum(sizes)

    # History starts empty for every independent classification.
    labels = _classify_history(
        case_codes,
        actor_codes,
        times,
        sorted_times,
        positions,
        offsets,
    )

    i = np.flatnonzero(labels >= 0)
    j = i + 1
    behavior_names = np.array(["C", "I", "HI", "HB"])

    return pd.DataFrame({
        "case_id": case_ids[i],
        "event_i": event_names[i],
        "timestamp_i": timestamps[i],
        "event_j": event_names[j],
        "timestamp_j": timestamps[j],
        "actor_i": actors[i],
        "actor_j": actors[j],
        "behavior": behavior_names[labels[i]],
        "delta_t": (
            (timestamps[j] - timestamps[i])
            / np.timedelta64(1, "s")
        ),
    })


print("Upload the saved BPIC2011 daily time-series CSV.")
daily_upload = files.upload()
daily_files = [
    name for name in daily_upload
    if name.lower().endswith(".csv")
]

if len(daily_files) != 1:
    raise ValueError("Upload exactly one daily time-series CSV.")

daily = pd.read_csv(
    daily_files[0], index_col=0, parse_dates=True
)
daily.index = (
    pd.to_datetime(daily.index)
    .tz_localize(None)
    .normalize()
)

print("Upload the original BPIC2011 XES, XES.GZ, or ZIP event log.")
log_upload = files.upload()
log_files = [
    name for name in log_upload
    if is_log_file(name)
]

if len(log_files) != 1:
    raise ValueError(
        "Upload exactly one XES, XES.GZ, or ZIP file. "
        f"Found: {log_files}"
    )

log_path = log_files[0]

if log_path.lower().endswith(".zip"):
    with zipfile.ZipFile(log_path) as zf:
        members = [
            name for name in zf.namelist()
            if is_log_file(name)
            and not name.lower().endswith(".zip")
        ]

        if len(members) != 1:
            raise ValueError(
                "Expected exactly one XES/XES.GZ in ZIP; "
                f"found {members[:10]}"
            )

        log_path = str(
            Path("/content/bpic2011_cutoff_test")
            / Path(members[0]).name
        )
        Path(log_path).parent.mkdir(
            parents=True, exist_ok=True
        )

        with zf.open(members[0]) as src:
            with open(log_path, "wb") as dst:
                shutil.copyfileobj(src, dst)

print("Reading XES events...")
xes = pm4py.convert_to_dataframe(
    pm4py.read_xes(log_path)
)

CASE = "case:concept:name"
EVENT = "concept:name"
TIME = "time:timestamp"

ACTOR = next(
    (
        column
        for column in ["org:resource", "org:group"]
        if column in xes.columns
    ),
    None,
)

if ACTOR is None:
    raise KeyError(
        "XES needs an org:resource or org:group column."
    )

# Process timestamps remain timezone-aware for elapsed-time arithmetic.
process = xes[[CASE, EVENT, TIME, ACTOR]].copy()
process[TIME] = pd.to_datetime(
    process[TIME], errors="coerce", utc=True
)

process = process.dropna(
    subset=[CASE, TIME]
).copy()
process[CASE] = process[CASE].astype(str)

# Process daily grouping uses Amsterdam calendar dates.
process["event_day"] = (
    process[TIME]
    .dt.tz_convert("Europe/Amsterdam")
    .dt.tz_localize(None)
    .dt.normalize()
)

# Actor classification uses Amsterdam local clock dates.
actor_events = xes[[CASE, EVENT, TIME, ACTOR]].copy()
actor_events[TIME] = (
    pd.to_datetime(
        actor_events[TIME], errors="coerce", utc=True
    )
    .dt.tz_convert("Europe/Amsterdam")
    .dt.tz_localize(None)
)

actor_events = actor_events.dropna(
    subset=[CASE, EVENT, TIME, ACTOR]
).copy()
actor_events[CASE] = actor_events[CASE].astype(str)

actor_events = actor_events.rename(columns={
    CASE: "case_id",
    EVENT: "event",
    TIME: "timestamp",
    ACTOR: "actor",
})
actor_events["event"] = actor_events["event"].astype(str)
actor_events["actor"] = actor_events["actor"].astype(str)

# Sort once. Every cutoff filter preserves this order.
actor_events = actor_events.sort_values(
    ["case_id", "timestamp"], kind="mergesort"
).reset_index(drop=True)

print(f"Process events: {len(process):,}")
print(f"Actor-complete events: {len(actor_events):,}")

print("Compiling the classifier (first run only)...")
_classify_history(
    np.array([0, 0], dtype=np.int64),
    np.array([0, 0], dtype=np.int64),
    np.array([0, 1], dtype=np.int64),
    np.array([0, 1], dtype=np.int64),
    np.array([0, 1], dtype=np.int64),
    np.array([0, 2], dtype=np.int64),
)

print("Classifying the full XES log for the reference...")
full_transitions = classify_at_cutoff(actor_events)
print(f"Full-log transitions: {len(full_transitions):,}")

label_key = [
    "case_id",
    "event_i",
    "timestamp_i",
    "event_j",
    "timestamp_j",
    "actor_i",
    "actor_j",
]
behaviors = ["C", "I", "HI", "HB"]

full_labels = full_transitions[
    label_key + ["behavior"]
]
full_transition_days = pd.to_datetime(
    full_transitions["timestamp_j"]
).dt.normalize()

label_mismatches_total = 0
actor_mismatches_vs_full = {
    column: 0
    for behavior in behaviors
    for column in [
        f"Cum_Count_{behavior}",
        f"Cum_Time_{behavior}_seconds",
    ]
}
cutoff_rows = []

for day in tqdm(
    daily.index, desc="Rebuilding daily cutoffs"
):
    cutoff = day + pd.Timedelta(days=1)

    # Classify this cutoff independently using only available events.
    prefix = actor_events.loc[
        actor_events["timestamp"] < cutoff
    ]
    cutoff_transitions = classify_at_cutoff(prefix)

    # Compare labels against the full-log reference.
    if len(cutoff_transitions):
        label_check = cutoff_transitions[
            label_key + ["behavior"]
        ].merge(
            full_labels,
            on=label_key,
            how="left",
            suffixes=("_cutoff", "_full"),
        )

        label_mismatch = int(
            (
                label_check["behavior_full"].notna()
                & (
                    label_check["behavior_cutoff"]
                    != label_check["behavior_full"]
                )
            ).sum()
        )
    else:
        label_mismatch = 0

    label_mismatches_total += label_mismatch

    row = {
        "cutoff_date": day,
        "label_mismatches": label_mismatch,
    }

    cutoff_days = pd.to_datetime(
        cutoff_transitions["timestamp_j"]
    ).dt.normalize()

    full_available = full_transitions.loc[
        full_transition_days <= day
    ]

    for behavior in behaviors:
        cutoff_selected = cutoff_transitions.loc[
            (cutoff_transitions["behavior"] == behavior)
            & (cutoff_days <= day)
        ]

        full_selected = full_available.loc[
            full_available["behavior"] == behavior
        ]

        count_col = f"Cum_Count_{behavior}"
        time_col = f"Cum_Time_{behavior}_seconds"

        cutoff_count = len(cutoff_selected)
        full_count = len(full_selected)

        cutoff_seconds = (
            float(cutoff_selected["delta_t"].sum())
            if len(cutoff_selected)
            else 0.0
        )
        full_seconds = (
            float(full_selected["delta_t"].sum())
            if len(full_selected)
            else 0.0
        )

        row[count_col] = cutoff_count
        row[f"full_{count_col}"] = full_count
        row[time_col] = cutoff_seconds
        row[f"full_{time_col}"] = full_seconds

        if cutoff_count != full_count:
            actor_mismatches_vs_full[count_col] += 1

        if not np.isclose(
            cutoff_seconds, full_seconds, atol=1e-3
        ):
            actor_mismatches_vs_full[time_col] += 1

    cutoff_rows.append(row)

cutoff_results = pd.DataFrame(cutoff_rows)

# Rebuild process inputs using Amsterdam dates and elapsed time in days.
process["case_start"] = (
    process.groupby(CASE)[TIME].transform("min")
)

# Advance the calendar date first, then localize.
# This gives the next Amsterdam midnight across DST changes.
process["cutoff_utc"] = (
    (process["event_day"] + pd.Timedelta(days=1))
    .dt.tz_localize("Europe/Amsterdam")
    .dt.tz_convert("UTC")
)

process["elapsed_days_at_eod"] = (
    (process["cutoff_utc"] - process["case_start"])
    .dt.total_seconds()
    / 86400.0
)

case_day = process[
    [
        CASE,
        "event_day",
        "case_start",
        "elapsed_days_at_eod",
    ]
].drop_duplicates([CASE, "event_day"])

process_daily = process.groupby("event_day").agg(
    Events_During_Day=(EVENT, "size"),
    Active_Resources_During_Day=(ACTOR, "nunique"),
)
process_daily["Cases_With_Activity"] = (
    process.groupby("event_day")[CASE].nunique()
)
process_daily["Avg_Elapsed_Time"] = (
    case_day.groupby("event_day")[
        "elapsed_days_at_eod"
    ].mean()
)

process_mismatches = {}

for column in [
    "Events_During_Day",
    "Active_Resources_During_Day",
    "Cases_With_Activity",
    "Avg_Elapsed_Time",
]:
    if column in daily.columns:
        expected = process_daily[column].reindex(
            daily.index
        )
        actual = pd.to_numeric(
            daily[column], errors="coerce"
        )

        process_mismatches[column] = int(
            (
                ~np.isclose(
                    actual.fillna(0),
                    expected.fillna(0),
                    atol=1e-6,
                )
            ).sum()
        )

print("\nCUTOFF TEST RESULTS")
print(
    "Label mismatches against full-XES labels: "
    f"{label_mismatches_total:,}"
)
print(f"Daily dates tested: {len(daily):,}")

print("\nProcess-input mismatches against saved CSV:")
display(
    pd.DataFrame(
        [
            {
                "column": column,
                "mismatching_dates": count,
            }
            for column, count in process_mismatches.items()
        ],
        columns=["column", "mismatching_dates"],
    ).sort_values(
        "mismatching_dates", ascending=False
    )
)

print(
    "\nActor-feature mismatches: "
    "cutoff run vs full-XES reference"
)
display(
    pd.DataFrame([
        {
            "column": column,
            "mismatching_dates": count,
        }
        for column, count in actor_mismatches_vs_full.items()
    ]).sort_values(
        "mismatching_dates", ascending=False
    )
)

print("\nFirst cutoff results:")
display(cutoff_results.head(10))

all_checks_pass = (
    label_mismatches_total == 0
    and sum(process_mismatches.values()) == 0
    and sum(actor_mismatches_vs_full.values()) == 0
)

print("\nPASS" if all_checks_pass else "\nFAIL")

Upload the saved BPIC2011 daily time-series CSV.


Saving BPIC2011_final_online_daily_time_series_cutoff_safe_days.csv to BPIC2011_final_online_daily_time_series_cutoff_safe_days (1).csv
Upload the original BPIC2011 XES, XES.GZ, or ZIP event log.


Saving Hospital_log.xes.gz to Hospital_log.xes (1).gz
Reading XES events...


parsing log, completed traces ::   0%|          | 0/1143 [00:00<?, ?it/s]

Process events: 150,291
Actor-complete events: 150,275
Compiling the classifier (first run only)...
Classifying the full XES log for the reference...
Full-log transitions: 149,132


Rebuilding daily cutoffs:   0%|          | 0/1173 [00:00<?, ?it/s]

... (3 outputs left out)

,column,mismatching_dates
0,Cum_Count_C,0
1,Cum_Time_C_seconds,0
2,Cum_Count_I,0
3,Cum_Time_I_seconds,0
4,Cum_Count_HI,0
5,Cum_Time_HI_seconds,0
6,Cum_Count_HB,0
7,Cum_Time_HB_seconds,0



First cutoff results:


,cutoff_date,label_mismatches,Cum_Count_C,full_Cum_Count_C,Cum_Time_C_seconds,full_Cum_Time_C_seconds,Cum_Count_I,full_Cum_Count_I,Cum_Time_I_seconds,full_Cum_Time_I_seconds,Cum_Count_HI,full_Cum_Count_HI,Cum_Time_HI_seconds,full_Cum_Time_HI_seconds,Cum_Count_HB,full_Cum_Count_HB,Cum_Time_HB_seconds,full_Cum_Time_HB_seconds
0,2005-01-03,0,33,33,0.0,0.0,0,0,0.0,0.0,6,6,0.0,0.0,0,0,0.0,0.0
1,2005-01-04,0,33,33,0.0,0.0,0,0,0.0,0.0,6,6,0.0,0.0,0,0,0.0,0.0
2,2005-01-05,0,34,34,0.0,0.0,0,0,0.0,0.0,10,10,345600.0,345600.0,0,0,0.0,0.0
3,2005-01-06,0,86,86,0.0,0.0,0,0,0.0,0.0,21,21,345600.0,345600.0,1,1,259200.0,259200.0
4,2005-01-07,0,87,87,0.0,0.0,0,0,0.0,0.0,26,26,518400.0,518400.0,1,1,259200.0,259200.0
5,2005-01-08,0,87,87,0.0,0.0,0,0,0.0,0.0,26,26,518400.0,518400.0,1,1,259200.0,259200.0
6,2005-01-09,0,87,87,0.0,0.0,0,0,0.0,0.0,26,26,518400.0,518400.0,1,1,259200.0,259200.0
7,2005-01-10,0,144,144,0.0,0.0,0,0,0.0,0.0,41,41,1123200.0,1123200.0,1,1,259200.0,259200.0
8,2005-01-11,0,169,169,0.0,0.0,0,0,0.0,0.0,49,49,1123200.0,1123200.0,1,1,259200.0,259200.0
9,2005-01-12,0,287,287,0.0,0.0,0,0,0.0,0.0,76,76,2160000.0,2160000.0,1,1,259200.0,259200.0



PASS
